# 04 — Ventanado y extracción de features

**Versión robusta a memoria y NaN**

Extrae ventanas de ECG caso-por-caso, evalúa calidad a nivel de ventana,
interpola o descarta según criterios reproducibles, y guarda features para los notebooks 05 y 06.

**Pre-requisitos:**
1. `03_ecg_loading_and_visualization.ipynb` — descarga de señales `.npy`.
2. `03b_waveform_quality_eda.ipynb` (recomendado) — genera `valid_cases_flexible.csv`.

## Contexto metodológico

### Por qué no se descartan casos completos con NaN

El EDA técnico (03b) confirmó que prácticamente todos los archivos `.npy` contienen
algún NaN. Descartar casos completos por `nan_count > 0` eliminaría el dataset entero.
Los NaN son locales (artefactos transitorios, gaps de transmisión) y no invalidan el
caso. La calidad se evalúa a nivel de **ventana individual**.

### Por qué la limpieza se hace por ventana

Cada ventana de 2 s es independiente. Un artefacto en una parte de la señal no
contamina ventanas en otras partes. Filtrar por ventana maximiza los ejemplos de
entrenamiento disponibles.

### Por qué se usa `mmap_mode='r'`

El dataset ocupa ~25 GB de señales `.npy`. Cargar todo en RAM produce `MemoryError`.
Con `mmap_mode='r'` el archivo se mapea en disco: solo se leen las páginas
correspondientes a cada ventana accedida. El footprint de RAM se limita a un caso
a la vez.

### Criterios de calidad por ventana

| Criterio | Umbral | Acción |
|---|---|---|
| `nan_pct_window > 5%` | `MAX_NAN_PCT_WINDOW = 0.05` | Descartar ventana |
| `max_nan_gap > 0.5 s` | `MAX_NAN_GAP_SECONDS = 0.5` | Descartar ventana |
| NaN dentro del umbral | — | Interpolar linealmente |
| `std == 0` tras interpolación | — | Descartar (señal plana) |

### Conexión con notebooks 05 y 06

- **05**: carga `features_baseline.parquet`, entrena con `GroupKFold` por `case_id`.
- **06**: búsqueda de hiperparámetros sobre el mismo dataset.

`beat_type` **NO** es predictor. `rhythm_label` es el target.
`case_id` se conserva para el split por paciente (evita leakage).

In [3]:
import sys
import re
import warnings
from pathlib import Path

# Detectar PROJECT_ROOT desde el directorio del notebook
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from src import config
from src.data_loading import load_annotations_for_case
from src.preprocessing import apply_basic_filters
from src.windowing import build_windows_for_case
from src.features import compute_time_features, compute_per_beat_rr_features, compute_rr_features
from src.utils import get_logger

logger = get_logger("nb04")
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
print("Setup OK. PROJECT_ROOT:", PROJECT_ROOT)

Setup OK. PROJECT_ROOT: C:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml


## 1. Parámetros de configuración

In [4]:
# ---- Control de corrida ----
# None = corrida completa; usa 3 o 10 para pruebas rápidas
MAX_CASES: int | None = None

# ---- Parámetros de señal y ventanado ----
FS_HZ: int = config.DEFAULT_ECG_FS_HZ           # 500 Hz
WINDOW_SECONDS: float = config.DEFAULT_WINDOW_SECONDS  # 2.0 s
OVERLAP: float = config.DEFAULT_WINDOW_OVERLAP   # 0.0

# ---- Umbrales de calidad por ventana ----
MAX_NAN_PCT_WINDOW: float = 0.05    # descartar si >5% de muestras son NaN
MAX_NAN_GAP_SECONDS: float = 0.5    # descartar si brecha continua de NaN > 0.5 s

print(f"FS_HZ={FS_HZ} Hz | WINDOW_SECONDS={WINDOW_SECONDS}s | OVERLAP={OVERLAP}")
print(f"Muestras por ventana: {int(FS_HZ * WINDOW_SECONDS)}")
print(f"Umbral NaN pct  : <={MAX_NAN_PCT_WINDOW:.0%}")
print(f"Umbral NaN gap  : <={MAX_NAN_GAP_SECONDS}s = {int(MAX_NAN_GAP_SECONDS * FS_HZ)} muestras")

FS_HZ=500 Hz | WINDOW_SECONDS=2.0s | OVERLAP=0.0
Muestras por ventana: 1000
Umbral NaN pct  : <=5%
Umbral NaN gap  : <=0.5s = 250 muestras


In [5]:
from pathlib import Path
from src import config
import os

print("CWD actual:")
print(Path.cwd())

print("\nPROJECT ROOT estimado:")
print(Path.cwd().resolve())

print("\nconfig.VITALDB_WAVEFORMS_DIR:")
print(config.VITALDB_WAVEFORMS_DIR)

print("\n¿Existe carpeta waveforms?")
print(config.VITALDB_WAVEFORMS_DIR.exists())

print("\nArchivos .npy encontrados en config.VITALDB_WAVEFORMS_DIR:")
wave_files = list(config.VITALDB_WAVEFORMS_DIR.glob("case_*.npy"))
print(len(wave_files))
print(wave_files[:5])

print("\nContenido de data/raw:")
raw_dir = config.RAW_DATA_DIR if hasattr(
    config, "RAW_DATA_DIR") else config.DATA_DIR / "raw"
print(raw_dir)
print(list(raw_dir.iterdir()) if raw_dir.exists() else "No existe raw_dir")

CWD actual:
c:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml\notebooks

PROJECT ROOT estimado:
C:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml\notebooks

config.VITALDB_WAVEFORMS_DIR:
C:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml\data\raw\vitaldb_waveforms

¿Existe carpeta waveforms?
True

Archivos .npy encontrados en config.VITALDB_WAVEFORMS_DIR:
482
[WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_1001.npy'), WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_1002.npy'), WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_1018.npy'), WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-ar

## 2. Selección de casos

Se prioriza `valid_cases_flexible.csv` generado por el notebook 03b.
Si no existe, se detectan casos disponibles desde los archivos `.npy` en disco.

In [6]:
import re
from pathlib import Path

# Carpeta real de señales ECG descargadas
waveform_dir = config.VITALDB_WAVEFORMS_DIR

print("waveform_dir:", waveform_dir)
print("¿Existe?:", waveform_dir.exists())

wave_files = sorted(waveform_dir.glob("case_*.npy"))

print("Archivos .npy encontrados:", len(wave_files))
print("Primeros 5 archivos:", wave_files[:5])

available_case_ids = []

for path in wave_files:
    match = re.match(r"case_(\d+)\.npy$", path.name)
    if match:
        available_case_ids.append(int(match.group(1)))

available_case_ids = sorted(available_case_ids)

if MAX_CASES is not None:
    available_case_ids = available_case_ids[:MAX_CASES]

print("# casos disponibles:", len(available_case_ids))
print("Primeros 10 case_id:", available_case_ids[:10])

if not available_case_ids:
    raise FileNotFoundError(
        f"No hay casos disponibles en {waveform_dir}. "
        "Revisa que existan archivos case_<id>.npy."
    )

waveform_dir: C:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml\data\raw\vitaldb_waveforms
¿Existe?: True
Archivos .npy encontrados: 482
Primeros 5 archivos: [WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_1001.npy'), WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_1002.npy'), WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_1018.npy'), WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_1023.npy'), WindowsPath('C:/Users/juanc/OneDrive/Documentos/Doctorado/Cursos/Curso machine/vitaldb-arrhythmia-ml/data/raw/vitaldb_waveforms/case_103.npy')]
# casos disponibles: 482
Primeros 10 case_id: [12, 13, 19, 42, 96, 98, 103, 110

In [7]:
# ---------- diagnostico de directorio ----------
waveform_dir = config.VITALDB_WAVEFORMS_DIR
print(f"waveform_dir : {waveform_dir}")
print(f"Existe       : {waveform_dir.exists()}")

# 1. Fuente de verdad: archivos .npy presentes en disco
wave_files = sorted(waveform_dir.glob("case_*.npy"))
disk_case_ids = []
for p in wave_files:
    m = re.match(r"case_(\d+)\.npy$", p.name)
    if m:
        disk_case_ids.append(int(m.group(1)))

print(f"Archivos .npy en disco  : {len(disk_case_ids)}")
print(f"Primeros 5 archivos     : {[p.name for p in wave_files[:5]]}")

# 2. Intentar cruzar con valid_cases_flexible.csv
flexible_path = config.PROCESSED_DIR / "valid_cases_flexible.csv"
print()
print(f"valid_cases_flexible.csv existe: {flexible_path.exists()}")

if flexible_path.exists():
    try:
        valid_df = pd.read_csv(flexible_path)
        id_col = "case_id" if "case_id" in valid_df.columns else valid_df.columns[0]
        flexible_ids = set(valid_df[id_col].dropna().astype(int).tolist())
        # Interseccion: solo casos presentes en el CSV que ademas existen en disco
        available_case_ids = sorted(cid for cid in disk_case_ids if cid in flexible_ids)
        print(f"Fuente: valid_cases_flexible.csv interseccion disco ({len(available_case_ids)} casos)")
        if not available_case_ids:
            print("AVISO: interseccion vacia. Usando todos los .npy del disco como fallback.")
            available_case_ids = disk_case_ids
            print(f"Fuente fallback: todos los .npy en disco ({len(available_case_ids)} casos)")
    except Exception as exc:
        print(f"Error leyendo valid_cases_flexible.csv: {exc}")
        print("Usando todos los .npy del disco como fallback.")
        available_case_ids = disk_case_ids
else:
    available_case_ids = disk_case_ids
    print("NOTA: valid_cases_flexible.csv no encontrado. Considera ejecutar 03b primero.")
    print(f"Fuente: todos los .npy en disco ({len(available_case_ids)} casos)")

if not available_case_ids:
    raise FileNotFoundError(
        f"No hay archivos case_*.npy en {waveform_dir}. "
        "Asegurate de haber ejecutado el notebook 03 para descargar las senales."
    )

if MAX_CASES is not None:
    available_case_ids = available_case_ids[:MAX_CASES]
    print(f"MAX_CASES={MAX_CASES}: limitando a {len(available_case_ids)} casos.")

print()
print(f"Total de casos a procesar : {len(available_case_ids)}")
print(f"Primeros 10 case_id       : {available_case_ids[:10]}")

waveform_dir : C:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml\data\raw\vitaldb_waveforms
Existe       : True
Archivos .npy en disco  : 482
Primeros 5 archivos     : ['case_1001.npy', 'case_1002.npy', 'case_1018.npy', 'case_1023.npy', 'case_103.npy']

valid_cases_flexible.csv existe: True
Fuente: valid_cases_flexible.csv interseccion disco (482 casos)

Total de casos a procesar : 482
Primeros 10 case_id       : [12, 13, 19, 42, 96, 98, 103, 110, 114, 146]


## 3. Funciones de calidad por ventana

Evaluación de calidad y decisión de descarte/interpolación para cada ventana individual.

In [8]:
def _max_consecutive_nan_run(nan_mask: np.ndarray) -> int:
    # Longitud máxima de un bloque continuo de True en nan_mask.
    if not nan_mask.any():
        return 0
    x = nan_mask.astype(np.int8)
    d = np.diff(x, prepend=np.int8(0), append=np.int8(0))
    starts = np.where(d == 1)[0]
    ends = np.where(d == -1)[0]
    if len(starts) == 0:
        return 0
    return int((ends - starts).max())


def assess_window_quality(
    window: np.ndarray,
    fs_hz: int,
    max_nan_pct: float = MAX_NAN_PCT_WINDOW,
    max_nan_gap_seconds: float = MAX_NAN_GAP_SECONDS,
) -> dict:
    # Evalúa calidad de una ventana ECG y decide descarte o interpolación.
    # Retorna: nan_count_window, nan_pct_window, max_consecutive_nan,
    # max_nan_gap_seconds, inf_count_window, signal_std_window,
    # quality_status ('ok'|'interpolated'|'discarded'), quality_reason, clean_window.
    w = np.asarray(window, dtype=float).ravel()

    # Convertir Inf a NaN (copia para no modificar el array original)
    inf_mask = np.isinf(w)
    inf_count = int(inf_mask.sum())
    if inf_count > 0:
        w = w.copy()
        w[inf_mask] = np.nan

    nan_mask = np.isnan(w)
    nan_count = int(nan_mask.sum())
    nan_pct = float(nan_count) / len(w)
    max_consec = _max_consecutive_nan_run(nan_mask)
    max_gap_s = float(max_consec) / fs_hz
    signal_std = float(np.nanstd(w)) if nan_count < len(w) else 0.0

    metrics = {
        "nan_count_window":    nan_count,
        "nan_pct_window":      round(nan_pct, 6),
        "max_consecutive_nan": max_consec,
        "max_nan_gap_seconds": round(max_gap_s, 6),
        "inf_count_window":    inf_count,
        "signal_std_window":   round(signal_std, 6),
    }

    def _discard(reason: str) -> dict:
        return {**metrics, "quality_status": "discarded", "quality_reason": reason, "clean_window": None}

    # Criterios de descarte
    if nan_pct > max_nan_pct:
        return _discard("nan_pct_exceeded")
    if max_gap_s > max_nan_gap_seconds:
        return _discard("nan_gap_exceeded")

    # Interpolación si hay NaN/Inf dentro del umbral
    if nan_count > 0 or inf_count > 0:
        w_clean = pd.Series(w).interpolate(method="linear", limit_direction="both").to_numpy()
        if np.any(np.isnan(w_clean)):
            return _discard("interpolation_failed")
        if float(np.std(w_clean)) == 0.0:
            return _discard("zero_variance_after_interpolation")
        return {
            **metrics,
            "quality_status": "interpolated",
            "quality_reason": "ok_after_interpolation",
            "clean_window": w_clean,
        }

    # Ventana limpia — verificar varianza
    if float(np.std(w)) == 0.0:
        return _discard("zero_variance")

    return {
        **metrics,
        "quality_status": "ok",
        "quality_reason": "clean",
        "clean_window": w,
    }


print("Funciones de calidad definidas.")
print(f"  Umbral nan_pct : {MAX_NAN_PCT_WINDOW:.0%}")
print(f"  Umbral nan_gap : {MAX_NAN_GAP_SECONDS}s = {int(MAX_NAN_GAP_SECONDS * FS_HZ)} muestras @ {FS_HZ} Hz")

Funciones de calidad definidas.
  Umbral nan_pct : 5%
  Umbral nan_gap : 0.5s = 250 muestras @ 500 Hz


## 4. Procesamiento caso-por-caso

Para cada `case_id`:
1. Carga la señal con `mmap_mode='r'` (no ocupa RAM completa).
2. Filtra anotaciones con `apply_basic_filters`.
3. Construye ventanas centradas en latidos.
4. Evalúa calidad por ventana; descarta o interpola.
5. Extrae features temporales y RR sobre ventanas limpias.
6. Libera señal y ventanas de RAM antes de pasar al siguiente caso.

In [9]:
all_case_dfs: list = []
quality_log: list = []

n_skip_no_file = 0
n_skip_no_annot = 0
n_skip_no_beats = 0
n_skip_no_windows = 0

for cid in tqdm(available_case_ids, desc="Casos"):

    # 1. Verificar archivo de señal
    wave_path = config.VITALDB_WAVEFORMS_DIR / f"case_{cid}.npy"
    if not wave_path.exists():
        n_skip_no_file += 1
        continue

    # 2. Cargar y filtrar anotaciones
    try:
        beats_raw = load_annotations_for_case(cid)
    except FileNotFoundError:
        n_skip_no_annot += 1
        continue

    beats = apply_basic_filters(beats_raw)
    if beats.empty or config.BEAT_TIME_COLUMN not in beats.columns:
        n_skip_no_beats += 1
        continue

    # 3. Features RR globales del caso (replicadas en todas las ventanas del caso)
    rr_global = compute_rr_features(beats[config.BEAT_TIME_COLUMN].to_numpy())

    # 4. Features RR locales por latido (índice posicional 0..n_beats-1)
    rr_local_df = compute_per_beat_rr_features(beats[config.BEAT_TIME_COLUMN].to_numpy())
    # Mapeo: label del índice original → posición en rr_local_df
    orig_idx_to_pos = {lbl: pos for pos, lbl in enumerate(beats.index.tolist())}

    # 5. Cargar señal via mmap (no carga el archivo completo en RAM)
    signal = np.load(wave_path, mmap_mode="r")

    # 6. Construir ventanas centradas en latidos anotados
    windows, specs = build_windows_for_case(
        signal=signal,
        beats=beats,
        case_id=cid,
        fs_hz=FS_HZ,
        window_seconds=WINDOW_SECONDS,
        overlap=OVERLAP,
    )
    del signal  # liberar referencia mmap; GC cerrará el file handle

    if len(specs) == 0:
        n_skip_no_windows += 1
        del windows
        continue

    # 7. Evaluar calidad y extraer features por ventana
    case_rows: list = []

    for i, (win_data, spec) in enumerate(zip(windows, specs)):
        quality = assess_window_quality(win_data, FS_HZ)

        quality_log.append({
            "case_id":             cid,
            "window_id":           i,
            "beat_index":          spec.beat_index,
            "label":               spec.label,
            "quality_status":      quality["quality_status"],
            "quality_reason":      quality["quality_reason"],
            "nan_pct_window":      quality["nan_pct_window"],
            "max_nan_gap_seconds": quality["max_nan_gap_seconds"],
            "inf_count_window":    quality["inf_count_window"],
            "signal_std_window":   quality["signal_std_window"],
        })

        if quality["quality_status"] == "discarded":
            continue

        clean_win = quality["clean_window"]

        # Features temporales sobre la ventana limpia
        time_feats = compute_time_features(clean_win)

        # Features RR locales para este latido
        pos = orig_idx_to_pos.get(spec.beat_index)
        if pos is not None and pos < len(rr_local_df):
            rr_local = rr_local_df.iloc[pos].to_dict()
        else:
            rr_local = {"rr_prev": np.nan, "rr_next": np.nan,
                        "rr_mean_local": np.nan, "rr_ratio": np.nan}

        row: dict = {
            config.CASE_ID_COLUMN: cid,
            "window_id":           i,
            "beat_index":          spec.beat_index,
            "start_sample":        spec.start_sample,
            "end_sample":          spec.end_sample,
            "start_time":          round(spec.start_sample / FS_HZ, 4),
            "end_time":            round(spec.end_sample   / FS_HZ, 4),
            config.TARGET_COLUMN:  spec.label,
            # Métricas de calidad de la ventana
            "original_nan_pct":    quality["nan_pct_window"],
            "max_nan_gap_seconds": quality["max_nan_gap_seconds"],
            "was_interpolated":    quality["quality_status"] == "interpolated",
            "quality_status":      quality["quality_status"],
            "quality_reason":      quality["quality_reason"],
        }
        row.update(time_feats)
        row.update(rr_local)
        # Prefijo 'case_' para distinguir RR globales de RR locales
        row.update({f"case_{k}": v for k, v in rr_global.items()})

        case_rows.append(row)

    del windows  # liberar array de ventanas del caso

    if case_rows:
        all_case_dfs.append(pd.DataFrame(case_rows))

print()
print("=" * 52)
print(f"Casos con ventanas válidas     : {len(all_case_dfs)}")
print(f"Omitidos (sin archivo .npy)    : {n_skip_no_file}")
print(f"Omitidos (sin anotaciones)     : {n_skip_no_annot}")
print(f"Omitidos (sin latidos válidos) : {n_skip_no_beats}")
print(f"Omitidos (sin ventanas)        : {n_skip_no_windows}")

Casos:   0%|          | 0/482 [00:00<?, ?it/s]


Casos con ventanas válidas     : 481
Omitidos (sin archivo .npy)    : 0
Omitidos (sin anotaciones)     : 0
Omitidos (sin latidos válidos) : 0
Omitidos (sin ventanas)        : 0


## 5. Concatenar resultados y verificar integridad

In [10]:
if not all_case_dfs:
    raise RuntimeError(
        "No se generaron ventanas válidas. Verifica que existan archivos .npy "
        "y anotaciones, y revisa los umbrales de calidad."
    )

features_df = pd.concat(all_case_dfs, ignore_index=True)
del all_case_dfs  # liberar la lista de DataFrames intermedios

print(f"Dataset: {features_df.shape[0]:,} ventanas x {features_df.shape[1]} columnas")
print(f"Casos únicos: {features_df[config.CASE_ID_COLUMN].nunique()}")
print(f"\nDistribución {config.TARGET_COLUMN}:")
print(features_df[config.TARGET_COLUMN].value_counts().to_string())

# Eliminar filas sin etiqueta de ritmo
n_before = len(features_df)
features_df = features_df.dropna(subset=[config.TARGET_COLUMN])
features_df = features_df[
    ~features_df[config.TARGET_COLUMN].astype(str).str.strip().str.lower().isin({"nan", "none", ""})
].copy()
n_after = len(features_df)
if n_before != n_after:
    print(f"\nEliminadas {n_before - n_after} filas sin etiqueta válida.")

# Verificar NaN en features numéricas
_non_feat_cols = {
    config.CASE_ID_COLUMN, config.TARGET_COLUMN, config.BEAT_TYPE_COLUMN,
    config.SIGNAL_QUALITY_COLUMN,
    "window_id", "beat_index", "start_sample", "end_sample", "start_time", "end_time",
    "original_nan_pct", "max_nan_gap_seconds", "was_interpolated",
    "quality_status", "quality_reason",
}
_feature_cols = [c for c in features_df.columns if c not in _non_feat_cols]
nan_in_feats = features_df[_feature_cols].select_dtypes(include=[np.number]).isna().sum()
nan_in_feats = nan_in_feats[nan_in_feats > 0]
if nan_in_feats.empty:
    print("\nOK: sin NaN en columnas numéricas de features.")
else:
    print(f"\nATENCION: {len(nan_in_feats)} features con NaN (imputadas por Pipeline en nb05):")
    print(nan_in_feats.to_string())

features_df.head(3)

Dataset: 639,504 ventanas x 39 columnas
Casos únicos: 481

Distribución rhythm_label:
rhythm_label
N                               391877
AFIB/AFL                        158473
Patterned Ventricular Ectopy     23902
SND                              22224
Patterned Atrial Ectopy          19946
WAP/MAT                          10047
SVTA                              6396
AVB                               4193
VT                                1573
Unclassifiable                      59

Eliminadas 814 filas sin etiqueta válida.

ATENCION: 3 features con NaN (imputadas por Pipeline en nb05):
rr_prev     477
rr_next     479
rr_ratio    964


,case_id,window_id,beat_index,start_sample,end_sample,start_time,end_time,rhythm_label,original_nan_pct,max_nan_gap_seconds,...,rr_next,rr_mean_local,rr_ratio,case_rr_count,case_rr_mean,case_rr_std,case_rr_min,case_rr_max,case_rr_rmssd,case_rr_pnn50
0,12,0,0,4320811,4321811,8641.622,8643.622,N,0.0,0.0,...,0.744444,0.744444,NaN,1107,0.948409,0.933143,0.533333,20.055556,1.327557,0.254973
1,12,1,1,4321183,4322183,8642.366,8644.366,N,0.0,0.0,...,0.755556,0.750000,0.985294,1107,0.948409,0.933143,0.533333,20.055556,1.327557,0.254973
2,12,2,2,4321561,4322561,8643.122,8645.122,N,0.0,0.0,...,0.755556,0.755556,1.000000,1107,0.948409,0.933143,0.533333,20.055556,1.327557,0.254973


## 6. Reporte de calidad de ventanas

In [11]:
quality_df = pd.DataFrame(quality_log)
del quality_log  # liberar memoria

total_attempted = len(quality_df)
total_kept      = int((quality_df["quality_status"] != "discarded").sum())
total_discarded = int((quality_df["quality_status"] == "discarded").sum())
total_interp    = int((quality_df["quality_status"] == "interpolated").sum())
pct_disc        = 100.0 * total_discarded / max(total_attempted, 1)

print("=" * 60)
print(f"Total ventanas intentadas   : {total_attempted:>10,}")
print(f"Total ventanas mantenidas   : {total_kept:>10,}  ({100 - pct_disc:.1f}%)")
print(f"  - limpias (ok)            : {int((quality_df['quality_status'] == 'ok').sum()):>10,}")
print(f"  - interpoladas            : {total_interp:>10,}")
print(f"Total ventanas descartadas  : {total_discarded:>10,}  ({pct_disc:.1f}%)")
print("=" * 60)

disc = quality_df[quality_df["quality_status"] == "discarded"]
kept_q = quality_df[quality_df["quality_status"] != "discarded"]

print("\n--- Descartadas por razón ---")
if len(disc):
    print(disc["quality_reason"].value_counts().to_string())
else:
    print("(ninguna)")

print("\n--- Mantenidas por etiqueta de ritmo ---")
print(kept_q["label"].value_counts().to_string())

print("\n--- Descartadas por etiqueta de ritmo ---")
if len(disc):
    print(disc["label"].value_counts().to_string())
else:
    print("(ninguna)")

Total ventanas intentadas   :    640,274
Total ventanas mantenidas   :    639,504  (99.9%)
  - limpias (ok)            :    639,500
  - interpoladas            :          4
Total ventanas descartadas  :        770  (0.1%)

--- Descartadas por razón ---
quality_reason
nan_pct_exceeded    770

--- Mantenidas por etiqueta de ritmo ---
label
N                               391877
AFIB/AFL                        158473
Patterned Ventricular Ectopy     23902
SND                              22224
Patterned Atrial Ectopy          19946
WAP/MAT                          10047
SVTA                              6396
AVB                               4193
VT                                1573
Unclassifiable                      59

--- Descartadas por etiqueta de ritmo ---
label
N           746
SVTA         17
AFIB/AFL      7


## 7. Persistencia

Salidas de este notebook (excluidas por `.gitignore`, no se versionan):
- `data/processed/features_baseline.parquet` — dataset de features para modelado.
- `data/processed/window_quality_report.csv` — log de calidad por ventana.
- `data/processed/windowing_summary.csv` — resumen estadístico del proceso.

In [12]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# features_baseline: intentar parquet con pyarrow, luego fastparquet, luego CSV
out_parquet = config.PROCESSED_DIR / "features_baseline.parquet"
saved_as = None
for engine in ["pyarrow", "fastparquet"]:
    try:
        features_df.to_parquet(out_parquet, index=False, engine=engine)
        saved_as = str(out_parquet)
        print(f"Guardado ({engine}): {out_parquet}")
        break
    except Exception as exc:
        print(f"Parquet/{engine} falló: {exc}")

if saved_as is None:
    out_csv = config.PROCESSED_DIR / "features_baseline.csv"
    features_df.to_csv(out_csv, index=False)
    saved_as = str(out_csv)
    print(f"Guardado (CSV fallback): {out_csv}")

# Reporte de calidad por ventana
out_quality = config.PROCESSED_DIR / "window_quality_report.csv"
quality_df.to_csv(out_quality, index=False)
print(f"Guardado: {out_quality}")

# Resumen de ventanado
summary_data = {
    "total_cases_attempted":         [len(available_case_ids)],
    "total_cases_with_windows":      [features_df[config.CASE_ID_COLUMN].nunique()],
    "total_windows_attempted":       [total_attempted],
    "total_windows_kept":            [total_kept],
    "total_windows_discarded":       [total_discarded],
    "total_windows_interpolated":    [total_interp],
    "pct_windows_discarded":         [round(pct_disc, 2)],
    "features_output":               [saved_as],
    "max_nan_pct_threshold":         [MAX_NAN_PCT_WINDOW],
    "max_nan_gap_seconds_threshold": [MAX_NAN_GAP_SECONDS],
    "window_seconds":                [WINDOW_SECONDS],
    "fs_hz":                         [FS_HZ],
    "overlap":                       [OVERLAP],
    "max_cases_param":               [str(MAX_CASES)],
}
out_summary = config.PROCESSED_DIR / "windowing_summary.csv"
pd.DataFrame(summary_data).to_csv(out_summary, index=False)
print(f"Guardado: {out_summary}")

Guardado (pyarrow): C:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml\data\processed\features_baseline.parquet
Guardado: C:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml\data\processed\window_quality_report.csv
Guardado: C:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml\data\processed\windowing_summary.csv


## 8. Verificación de compatibilidad con notebook 05

In [13]:
print("=" * 60)
print("Compatibilidad con 05_baseline_modeling.ipynb")
print("=" * 60)

# Columnas clave requeridas por nb05
required_cols = [config.TARGET_COLUMN, config.CASE_ID_COLUMN]
missing_req = [c for c in required_cols if c not in features_df.columns]
if missing_req:
    print(f"ERROR: faltan columnas para nb05: {missing_req}")
else:
    print(f"OK: columnas clave presentes: {required_cols}")

# beat_type no debe ser predictor
if config.BEAT_TYPE_COLUMN in features_df.columns:
    print(f"INFO: '{config.BEAT_TYPE_COLUMN}' presente (informativa, NO predictora).")
else:
    print(f"OK: '{config.BEAT_TYPE_COLUMN}' ausente del dataset.")

# Features disponibles para nb05
_non_feat_nb05 = set(config.FORBIDDEN_FEATURE_COLUMNS) | {
    "window_id", "beat_index", "start_sample", "end_sample", "start_time", "end_time",
    "original_nan_pct", "max_nan_gap_seconds", "was_interpolated",
    "quality_status", "quality_reason",
}
feature_cols_nb05 = [c for c in features_df.columns if c not in _non_feat_nb05]
print(f"\nFeatures disponibles para nb05 ({len(feature_cols_nb05)}):")
print(feature_cols_nb05)

print(f"\nShape final: {features_df.shape}")
print(f"Clases unicas ({config.TARGET_COLUMN}): {sorted(features_df[config.TARGET_COLUMN].unique())}")
print(f"Casos unicos  ({config.CASE_ID_COLUMN}): {features_df[config.CASE_ID_COLUMN].nunique()}")

Compatibilidad con 05_baseline_modeling.ipynb
OK: columnas clave presentes: ['rhythm_label', 'case_id']
OK: 'beat_type' ausente del dataset.

Features disponibles para nb05 (26):
['mean', 'std', 'var', 'min', 'max', 'range', 'median', 'p25', 'p75', 'iqr', 'skew', 'kurtosis', 'energy', 'zero_crossing_rate', 'abs_mean', 'rr_prev', 'rr_next', 'rr_mean_local', 'rr_ratio', 'case_rr_count', 'case_rr_mean', 'case_rr_std', 'case_rr_min', 'case_rr_max', 'case_rr_rmssd', 'case_rr_pnn50']

Shape final: (638690, 39)
Clases unicas (rhythm_label): ['AFIB/AFL', 'AVB', 'N', 'Patterned Atrial Ectopy', 'Patterned Ventricular Ectopy', 'SND', 'SVTA', 'Unclassifiable', 'VT', 'WAP/MAT']
Casos unicos  (case_id): 481


## Auditoría: case_ids faltantes en el dataset final

Compara los `case_id` esperados con los presentes en `features_df`.
Para cada faltante, recorre el pipeline paso a paso para determinar
dónde fue descartado.

**No modifica el dataset** — solo diagnostica y reporta.

In [14]:
# ================================================================
# AUDITORIA: que case_id falta y por que
# Requiere haber ejecutado todas las celdas anteriores
# ================================================================

cases_in_features = set(features_df[config.CASE_ID_COLUMN].astype(int).unique())
cases_expected    = set(int(c) for c in available_case_ids)
cases_missing     = sorted(cases_expected - cases_in_features)

print("=" * 62)
print("AUDITORIA DE CASOS FALTANTES")
print("=" * 62)
print(f"Casos esperados (available_case_ids) : {len(cases_expected)}")
print(f"Casos en features_df                 : {len(cases_in_features)}")
print(f"Casos faltantes                      : {len(cases_missing)}")

if not cases_missing:
    print("Todos los casos esperados estan en el dataset final.")
else:
    print(f"\ncase_id faltantes: {cases_missing}")
    print()

    # quality_df puede estar disponible si se ejecuto la celda de reporte
    try:
        _has_quality = isinstance(quality_df, pd.DataFrame) and len(quality_df) > 0
    except NameError:
        _has_quality = False
        print("AVISO: quality_df no disponible. Ejecuta la celda de reporte primero.")

    audit_rows = []

    for cid in cases_missing:
        row = {"case_id": cid, "reason": "desconocida", "detail": ""}

        # 1. Senial en disco
        wave_path = config.VITALDB_WAVEFORMS_DIR / f"case_{cid}.npy"
        if not wave_path.exists():
            row.update(reason="sin_archivo_npy",
                       detail=f"No existe {wave_path.name}")
            audit_rows.append(row)
            continue

        npy_mb = round(wave_path.stat().st_size / 1e6, 2)
        row["detail"] = f".npy={npy_mb} MB"

        # 2. Anotaciones
        try:
            beats_raw = load_annotations_for_case(cid)
        except FileNotFoundError as exc:
            row.update(reason="sin_anotaciones", detail=str(exc))
            audit_rows.append(row)
            continue

        row["detail"] += f" | beats_raw={len(beats_raw)}"

        # 3. Filtros basicos
        beats_f = apply_basic_filters(beats_raw)
        row["detail"] += f" | beats_filtrados={len(beats_f)}"

        if beats_f.empty:
            row.update(reason="sin_beats_tras_filtros")
            audit_rows.append(row)
            continue

        if config.BEAT_TIME_COLUMN not in beats_f.columns:
            row.update(reason="sin_columna_beat_time",
                       detail=row["detail"] + f" | cols={list(beats_f.columns)}")
            audit_rows.append(row)
            continue

        beat_labels = (beats_f[config.TARGET_COLUMN].value_counts().to_dict()
                       if config.TARGET_COLUMN in beats_f.columns else {})
        row["detail"] += f" | labels_beats={beat_labels}"

        # 4. Ventanas y calidad (desde quality_df en memoria)
        if _has_quality:
            cq = quality_df[quality_df["case_id"] == cid]
            row["detail"] += f" | ventanas_intentadas={len(cq)}"

            if len(cq) == 0:
                row.update(reason="sin_ventanas_en_senial",
                           detail=row["detail"] + " | sin ventanas extraibles de la senial")
            else:
                kept = cq[cq["quality_status"] != "discarded"]
                row["detail"] += f" | ventanas_ok={len(kept)}"

                if len(kept) == 0:
                    reasons_dict = cq["quality_reason"].value_counts().to_dict()
                    row.update(reason="todas_ventanas_descartadas",
                               detail=row["detail"] + f" | razones={reasons_dict}")
                else:
                    # 5. Labels en ventanas mantenidas
                    labels_ok  = kept["label"].dropna().unique().tolist()
                    null_count = int(kept["label"].isna().sum())
                    row["detail"] += f" | labels_ventanas={labels_ok} | sin_label={null_count}"

                    if not labels_ok:
                        row.update(reason="sin_label_valido",
                                   detail=row["detail"] + " | todas las ventanas sin etiqueta")
                    else:
                        row.update(reason="eliminado_en_dropna_o_concat",
                                   detail=row["detail"] + " | tiene ventanas validas; revisar concat/dropna")
        else:
            row.update(reason="quality_df_no_disponible")

        audit_rows.append(row)

    # Imprimir detalle por caso faltante
    for r in audit_rows:
        print(f"  case_id={r['case_id']:6}  razon: {r['reason']}")
        print(f"           detalle: {r['detail']}")
        print()

    # Resumen por razon
    audit_df = pd.DataFrame(audit_rows)
    print("-" * 62)
    print("Resumen por razon:")
    print(audit_df["reason"].value_counts().to_string())
    print("-" * 62)

    # Guardar reporte de auditoria
    out_audit = config.PROCESSED_DIR / "missing_cases_audit.csv"
    audit_df.to_csv(out_audit, index=False)
    print(f"\nAuditoria guardada en: {out_audit}")

AUDITORIA DE CASOS FALTANTES
Casos esperados (available_case_ids) : 482
Casos en features_df                 : 481
Casos faltantes                      : 1

case_id faltantes: [3170]

  case_id=  3170  razon: todas_ventanas_descartadas
           detalle: .npy=43.91 MB | beats_raw=797 | beats_filtrados=755 | labels_beats={'N': 738, 'SVTA': 17} | ventanas_intentadas=755 | ventanas_ok=0 | razones={'nan_pct_exceeded': 755}

--------------------------------------------------------------
Resumen por razon:
reason
todas_ventanas_descartadas    1
--------------------------------------------------------------

Auditoria guardada en: C:\Users\juanc\OneDrive\Documentos\Doctorado\Cursos\Curso machine\vitaldb-arrhythmia-ml\data\processed\missing_cases_audit.csv


## 9. Próximos pasos

- **05_baseline_modeling.ipynb**: carga `features_baseline.parquet` y entrena clasificadores
  con `GroupKFold` por `case_id` para evitar leakage por paciente.
- **06_full_modeling_hyperparameter_search.ipynb**: búsqueda de hiperparámetros sobre
  el mismo dataset.

**Reproducibilidad**: para cambiar parámetros, modifica `MAX_CASES` (pruebas rápidas)
o los umbrales `MAX_NAN_PCT_WINDOW` / `MAX_NAN_GAP_SECONDS` y re-ejecuta desde la
celda de parámetros.